# Activity 2 — Output format

This is Activity 2 of the LLM Observability Hackathon. Run the cells top-to-bottom in a fresh kernel.

## Purpose
Compare an unconstrained prose answer with a constrained JSON answer, then validate both with the same parser. The trace records the generation output and the validation result.

## What to expect
The unconstrained answer is prose; the validator rejects it. The constrained answer is a JSON object; the validator accepts it. The notebook creates the traces `activity-2-output-unconstrained` and `activity-2-output-constrained`.

## Bad practice
Ask for free-form prose and extract fields with a regex. The parser has to guess where the object starts and ends and fails unpredictably on prose.

## Good practice
Constrain the output to a JSON object with an explicit schema, then validate the parsed fields before downstream use.

## Why the good practice works
The schema makes the expected keys and types explicit. Validation fails with a specific reason instead of silently accepting a malformed value, and the trace records the exact output the validator saw.

## Deliverable
The two trace URLs and the validation results for both variants.


In [1]:
import json, re, time
from IPython.display import Markdown, display
from experiment_utils import (client, langfuse, MODEL, RUN_ID, print_summary,
                              propagate_attributes, record_trace)


In [2]:
def validate_booking_request(text):
    match = re.search(r"\{.*\}", text, re.S)
    if not match:
        return False, None, "no JSON object found"
    try:
        parsed = json.loads(match.group(0))
    except json.JSONDecodeError as exc:
        return False, None, f"invalid JSON: {exc}"
    if not isinstance(parsed, dict):
        return False, parsed, "top-level value must be an object"
    required = {"origin", "destination", "travel_date", "passengers", "cabin"}
    if set(parsed) != required:
        return False, parsed, f"keys must be exactly {sorted(required)}"
    if not isinstance(parsed["origin"], str) or not parsed["origin"]:
        return False, parsed, "origin must be a non-empty string"
    if not isinstance(parsed["destination"], str) or not parsed["destination"]:
        return False, parsed, "destination must be a non-empty string"
    if not isinstance(parsed["travel_date"], str) or not re.fullmatch(r"\d{4}-\d{2}-\d{2}", parsed["travel_date"]):
        return False, parsed, "travel_date must be YYYY-MM-DD"
    if not isinstance(parsed["passengers"], int) or isinstance(parsed["passengers"], bool):
        return False, parsed, "passengers must be an integer"
    if parsed["cabin"] not in {"economy", "premium"}:
        return False, parsed, "cabin must be economy or premium"
    return True, parsed, "valid"

def run_format(variant):
    trace_name = f"activity-2-output-{variant}"
    system = (
        "Summarize the customer's booking request in one sentence."
        if variant == "unconstrained"
        else "Return ONLY a JSON object with exactly these keys: origin, destination, travel_date, passengers, cabin. travel_date must be YYYY-MM-DD; passengers must be an integer; cabin must be economy or premium. No prose and no markdown fences."
    )
    request = "Book 2 passengers from Lisbon to Madrid on 2026-12-01, economy."
    with langfuse.start_as_current_observation(name=trace_name, as_type="span") as root:
        with propagate_attributes(
            session_id=f"activity-2-{RUN_ID}",
            tags=["activity-2", variant],
            trace_name=trace_name,
        ):
            messages = [{"role": "system", "content": system}, {"role": "user", "content": request}]
            started = time.time()
            response = client.chat.completions.create(
                model=MODEL, messages=messages, temperature=0, max_tokens=300,
                name="activity-2-generation",
            )
            latency = time.time() - started
            answer = response.choices[0].message.content
            valid, parsed, reason = validate_booking_request(answer)
            trace_id = langfuse.get_current_trace_id()
            root.update(output={"valid": valid, "reason": reason, "parsed": parsed})
    usage = response.usage
    return {
        "variant": variant,
        "answer": answer,
        "valid": valid,
        "parsed": parsed,
        "reason": reason,
        "input_tokens": getattr(usage, "prompt_tokens", None),
        "output_tokens": getattr(usage, "completion_tokens", None),
        "latency": latency,
        "trace_id": trace_id,
        "trace_url": record_trace(f"activity-2-{variant}", trace_id),
    }

runs = [run_format("unconstrained"), run_format("constrained")]
rows = ["| variant | valid? | reason | input tokens | output tokens | latency (s) | trace |",
        "|---|---|---|---|---|---|---|"]
for run in runs:
    reason = (run["reason"] or "").replace("|", "\\|")
    rows.append(f"| {run['variant']} | {run['valid']} | {reason} | {run['input_tokens']} | {run['output_tokens']} | {run['latency']:.1f} | {run['trace_url']} |")
display(Markdown("\n".join(rows)))

| variant | valid? | reason | input tokens | output tokens | latency (s) | trace |
|---|---|---|---|---|---|---|
| unconstrained | False | no JSON object found | 60 | 66 | 1.2 | http://localhost:3000/project/langfuse-observability/traces/dfd9c4bc71d4aec710ea9c753146c379 |
| constrained | True | valid | 102 | 65 | 1.0 | http://localhost:3000/project/langfuse-observability/traces/6b80eb28c53976ba7358d3de57f72aa2 |

## Wrap-up

Run this cell to print the session id and trace URLs for this activity, then flush the Langfuse client. Submit the trace URLs and your findings.


In [3]:
print_summary(2)
langfuse.flush()
print("flushed")

run id: 6afa1d57
session activity-2-6afa1d57
activity-2-unconstrained: http://localhost:3000/project/langfuse-observability/traces/dfd9c4bc71d4aec710ea9c753146c379
activity-2-constrained: http://localhost:3000/project/langfuse-observability/traces/6b80eb28c53976ba7358d3de57f72aa2
flushed
